In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from local_variables import original_datasets_files_path
from src.utils import load_config

from src.data.normalize_labels import LABEL_MAPPING_CLEANED_NAMES

In [2]:
n_samples = 1_000_000
min_required = 10000

In [3]:
def normalize_labels(df, class_col):
    # Map every label, and send anything unmapped to 'other'
    return (
        df[class_col]
        .map(LABEL_MAPPING_CLEANED_NAMES)
        .fillna("other")              # everything else → other
    )

In [4]:
# df_name = "cic_ids_2017"
# df_name = "cic_ton_iot"
# df_name = "cic_bot_iot"
# df_name = "cic_unsw"
df_name = "cic_ddos_2019"

In [5]:
df = pd.read_parquet(os.path.join(original_datasets_files_path, f"{df_name}.parquet"))
df.head()

,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Tot Fwd Pkts,Tot Bwd Pkts,...,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Attack
0,192.168.50.7-4.2.2.4-54035-53-17,192.168.50.7,54035,4.2.2.4,53,17,2018-12-01 10:52:37.371731,20684,2,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,BENIGN
1,192.168.50.6-4.2.2.4-65384-53-17,192.168.50.6,65384,4.2.2.4,53,17,2018-12-01 11:05:47.686812,20627,2,2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,BENIGN
2,192.168.50.254-224.0.0.5-0-0-0,192.168.50.254,0,224.0.0.5,0,0,2018-12-01 11:08:46.949838,4,3,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,BENIGN
3,172.217.10.2-192.168.50.8-443-59128-6,192.168.50.8,59128,172.217.10.2,443,6,2018-12-01 10:54:12.793019,0,2,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,BENIGN
4,172.217.10.14-192.168.50.6-80-57000-6,192.168.50.6,57000,172.217.10.14,80,6,2018-12-01 10:54:15.907166,32457,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,BENIGN


In [6]:
class_col = "Attack"
timestamp_col = "Timestamp"
flow_id_col = "Flow ID"

In [7]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(axis=0, how='any', inplace=True)
df.drop_duplicates(subset=list(set(
    df.columns) - set([timestamp_col, flow_id_col])), keep="first", inplace=True)

In [8]:
df.columns

Index(['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol',
       'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts',
       'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max',
       'Fwd Pkt Len Min', 'Fwd Pkt Len Mean', 'Fwd Pkt Len Std',
       'Bwd Pkt Len Max', 'Bwd Pkt Len Min', 'Bwd Pkt Len Mean',
       'Bwd Pkt Len Std', 'Flow Byts/s', 'Flow Pkts/s', 'Flow IAT Mean',
       'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Tot',
       'Fwd IAT Mean', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Tot',
       'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
       'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags',
       'Fwd Header Len', 'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s',
       'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std',
       'Pkt Len Var', 'FIN Flag Cnt', 'SYN Flag Cnt', 'RST Flag Cnt',
       'PSH Flag Cnt', 'ACK Flag Cnt', 'URG Flag Cnt', 'CWE Flag Count',
       'ECE Flag Cnt', 'Dow

In [9]:
df[class_col].value_counts()


Attack
TFTP             3823161
Syn              1170180
MSSQL            1116977
DrDoS_SNMP       1024153
DrDoS_DNS         974927
DrDoS_MSSQL       879122
DrDoS_NetBIOS     783331
UDP               758164
NetBIOS           698307
DrDoS_UDP         618863
DrDoS_SSDP        513699
DrDoS_LDAP        426480
LDAP              372811
DrDoS_NTP         239198
UDP-lag            65926
Portmap            35449
BENIGN             22173
UDPLag               375
WebDDoS               88
Name: count, dtype: int64

In [10]:
df[class_col] = normalize_labels(df, class_col)

In [11]:
df[class_col].value_counts()

Attack
tftp             3823161
syn              1170180
mssql            1116977
drdos_snmp       1024153
drdos_dns         974927
drdos_mssql       879122
drdos_netbios     783331
udp               758164
netbios           698307
drdos_udp         618863
drdos_ssdp        513699
drdos_ldap        426480
ldap              372811
drdos_ntp         239198
other              65926
portmap            35449
benign             22173
udplag               375
webddos               88
Name: count, dtype: int64

In [12]:
# df = df[~df[class_col].isin(['mitm', 'ddos', 'dos'])]

In [13]:
if len(df) >= n_samples:

    # Step 1: Count original class sizes
    class_counts = df[class_col].value_counts()

    # Step 2: Separate small and large classes
    small_classes = class_counts[class_counts < min_required].index
    large_classes = class_counts[class_counts >= min_required].index

    # Step 3: Include all rows from small classes (don’t sample)
    included_small = df[df[class_col].isin(small_classes)]

    # Step 4: Calculate proportions for large classes only
    large_class_counts = class_counts[large_classes]
    large_class_props = large_class_counts / large_class_counts.sum()

    # Step 5: Allocate remaining samples to large classes
    remaining_samples = n_samples - len(included_small)
    large_class_samples = (large_class_props * remaining_samples).astype(int)

    # Step 6: Sample from large classes proportionally
    sampled_large = pd.concat([
        df[df[class_col] == cls].sample(
            n=large_class_samples[cls],
            random_state=42
        )
        for cls in large_class_samples.index
    ])

    # Step 7: Combine and shuffle
    subset = pd.concat([sampled_large, included_small])
    subset = subset.sample(frac=1, random_state=42).reset_index(drop=True)
else:
    subset = df.copy()


In [14]:
# len(subset[class_col].unique())

In [15]:
subset[class_col].value_counts()

Attack
tftp             282586
syn               86493
mssql             82560
drdos_snmp        75699
drdos_dns         72061
drdos_mssql       64979
drdos_netbios     57899
udp               56039
netbios           51614
drdos_udp         45742
drdos_ssdp        37969
drdos_ldap        31522
ldap              27556
drdos_ntp         17680
other              4872
portmap            2620
benign             1638
udplag              375
webddos              88
Name: count, dtype: int64

In [16]:
len(subset)

999992

In [17]:
subset.to_parquet(f"{original_datasets_files_path}\\{df_name}_1m.parquet")